# Hyperparameter Optimization (HPO) — YOLO26-seg / ISIC 2018 Task 1

**Objetivo deste notebook**: explicar de forma reproduzível e didática o
processo de otimização de hiperparâmetros que usamos para fechar o gap entre
o melhor resultado anedótico (v2 *original* = 0.7368) e a baseline reproduzível
(v2 *re-run* = 0.7169) no fine-tuning do YOLO26-seg sobre o ISIC 2018 Task 1.

Este é um notebook de **apresentação**: cada seção começa com um bloco de
texto explicando o quê e o porquê antes do código. Os gráficos são otimizados
para projeção (fontes grandes, paleta colorblind-safe Okabe-Ito, eixos
cropados na zona de sinal).

---

## Sumário

1. **Contexto** — por que precisamos de HPO depois das v2-v7
2. **NAS vs HPO** — distinção formal e por que HPO foi a escolha
3. **Método** — algoritmo genético do Ultralytics (`model.tune()`)
4. **Espaço de busca** — quais hp variamos e por quê
5. **Convergência da busca** — o GA realmente melhorou ou estagnou em ruído?
6. **O que aprendemos** — correlação Pearson hp × fitness
7. **Hiperparâmetros campeões** — Δ vs default v7
8. **Validação em treino full-length** — o ganho sobreviveu às 120 ep?
9. **Comparação final** — v2-v7 vs v8 (HPO-tuned)
10. **Conclusões e próximos passos**


## Setup

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

warnings.filterwarnings('ignore')

# ---- estilo de apresentação --------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "legend.fontsize": 10,
    "font.size": 11,
})
sns.set_context("notebook", font_scale=1.05)

# ---- paleta Okabe-Ito (colorblind-safe) --------------------------------------
OK_BLUE       = "#0072B2"
OK_ORANGE     = "#E69F00"
OK_GREEN      = "#009E73"
OK_VERMILLION = "#D55E00"
OK_PURPLE     = "#CC79A7"
OK_SKY        = "#56B4E9"
OK_YELLOW     = "#F0E442"
OK_GREY       = "#999999"

# Versões + cores (mesmas do compare_versions p/ consistência cruzada)
VERSION_PALETTE = {
    "v1": OK_BLUE,
    "v2": OK_ORANGE,
    "v3": OK_GREEN,
    "v7": OK_VERMILLION,
    "v8": OK_PURPLE,
}

print("Setup OK. Paleta:", list(VERSION_PALETTE.keys()))


### Configuração de paths

Mude `unb_server` conforme o ambiente (`True` para o servidor do laboratório,
`False` para máquina local). O notebook tolera arquivos faltantes — você pode
rodá-lo mesmo sem todos os logs disponíveis.

In [ ]:
unb_server = True

if unb_server:
    BASE_DIR = Path("/home/antoniovinicius/projects/sandbox_yolo26")
else:
    BASE_DIR = Path("/home/avmoura_linux/Documents/unb/sandbox_yolo26")

# HPO output (uma rodada × small)
TUNE_DIR  = BASE_DIR / "logs" / "tune_isic_2018_task_1_small"
TUNE_CSV  = TUNE_DIR / "tune_results.csv"
TUNE_NDJ  = TUNE_DIR / "tune_results.ndjson"
BEST_YAML = TUNE_DIR / "best_hyperparameters.yaml"

# Treino full-length validando os hp campeões
# Aceita tanto a pasta antiga (gerada pelo train_with_tuned_hp.py) quanto a v8
TUNED_DIRS_CANDIDATES = [
    BASE_DIR / "logs" / "yolo26_small_ft_isic_2018_v7_tuned",
    BASE_DIR / "logs" / "train_isic_2018_task_1_v8" / "yolo26_small_ft_isic_2018_v8",
    BASE_DIR / "logs" / "yolo26_small_ft_isic_2018_v8",
]
TUNED_DIR = next((d for d in TUNED_DIRS_CANDIDATES if (d / "results.csv").exists()), TUNED_DIRS_CANDIDATES[0])

print("BASE_DIR :", BASE_DIR)
print("TUNE_DIR :", TUNE_DIR, "[OK]" if TUNE_CSV.exists() or TUNE_NDJ.exists() else "[FALTANDO]")
print("BEST yaml:", BEST_YAML, "[OK]" if BEST_YAML.exists() else "[FALTANDO]")
print("TUNED dir:", TUNED_DIR, "[OK]" if (TUNED_DIR / "results.csv").exists() else "[FALTANDO]")


## 1. Contexto — por que HPO depois das v2-v7

Antes do HPO, fizemos seis ablations manuais (v2, v3, v4, v5, v6, v7) variando
otimizador, learning rate, augmentação e schedule. Os resultados no `small`
(120 épocas, mesma seed, mesmo dataset):

| Versão | mAP50-95 (M) | O que mudou |
|---|---|---|
| v2 *original* | 0.7368 | run sortudo (não reproduzível) |
| v3 small | 0.7099 | mudanças compostas de augment/loss |
| v4 small | 0.6484 | augment ruidoso (mixup + copy_paste agressivos) |
| v5 small | 0.7021 | minimalista, AdamW lr0=1e-3 |
| v6 small | 0.6999 | = v5 com lr0 dobrado |
| v7 small | 0.6964 | otimizador MuSGD |
| v2 *re-run* | **0.7169** | mesma config v2, seed 0 — **baseline reproduzível** |

**Hipótese**: o ganho do v2 *original* não é exclusivo dele — existe um
**conjunto de hiperparâmetros melhor que o default**, e exploração manual
não foi capaz de encontrá-lo de forma sistemática.

**Limitação do approach manual**: cada experimento tem custo ~1-2h de GPU,
e o espaço é alto-dimensional (>20 hp). Variar um hp de cada vez ignora
interações (ex: lr0 ótimo depende de momentum, weight_decay, warmup, etc).

**Solução**: substituir tentativa-e-erro por busca automatizada.

## 2. NAS vs HPO — distinção formal

Os dois termos são frequentemente confundidos, mas significam coisas distintas:

### Neural Architecture Search (NAS)
Busca automatizada sobre a **arquitetura** do modelo: profundidade
(`depth_multiple`), largura (`width_multiple`), tipo e ordem dos blocos
(conv, residual, attention, MBConv, ...), kernel sizes, número de canais.
O resultado é um modelo com topologia diferente do original. Exemplos:
EfficientNet-NAS, NASNet, AmoebaNet.

### Hyperparameter Optimization (HPO)
Busca automatizada sobre os **hiperparâmetros de treino**: `lr0`, `momentum`,
`weight_decay`, intensidade de cada augmentation, pesos das losses, schedule.
A arquitetura permanece **fixa**; varia apenas *como* o modelo é treinado.

### Por que escolhemos HPO

1. **A arquitetura YOLO26-seg já é validada em COCO**. Modificar
   `depth_multiple` ou `width_multiple` invalidaria os pesos pré-treinados —
   perderíamos a maior vantagem do fine-tuning.
2. **Os experimentos manuais v2-v7 mostraram que o gargalo é hiperparâmetro**.
   Variando lr, optimizer e augmentação isoladamente, o `mAP50-95(M)` flutuou
   entre 0.65 e 0.74 sem mexer na arquitetura.
3. **Custo**. NAS de verdade requer treinar dezenas de candidatos do zero —
   ordem de centenas de horas. HPO reaproveita os pesos pré-treinados em todos
   os trials.
4. **Tamanho do dataset**. ISIC 2018 Task 1 tem ~2.6k imagens — busca
   arquitetural seria desproporcional para esse regime de dados.

> *Observação*: alguns trabalhos usam "NAS" de forma genérica para "otimizar
> a configuração do modelo". Tecnicamente, o que entregamos aqui é HPO.

## 3. Método — algoritmo genético do Ultralytics

O Ultralytics expõe um tuner embutido (`model.tune()`) que implementa um
**algoritmo evolutivo / genético (GA)** para HPO. Pseudocódigo:

```
Inicialização:
    sortear hp iniciais ~ Uniforme(SEARCH_SPACE)
    avaliar fitness do trial 0

Para iteração i = 1 .. N:
    1. treinar o modelo por E épocas curtas com hp_i
    2. fitness_i = combinação ponderada de mAP50 + mAP50-95 (box + mask)
    3. salvar (fitness_i, hp_i) em tune_results.csv

    4. selecionar top-K trials feitos até agora como "pais" elitistas
    5. mutar hp via Gaussiana centrada no melhor pai:
           hp_{i+1} = hp_pai + N(0, σ²) * (range_max - range_min)
       onde σ decai linearmente de 0.2 → 0.1 ao longo das primeiras 300 iter

Fim:
    salvar best_hyperparameters.yaml com a config top-1
```

### Por que GA e não outras técnicas

| Técnica | Quando faz sentido | No nosso caso |
|---|---|---|
| **Random search** | budget muito pequeno, baseline | inferior a GA em <50 trials |
| **Grid search** | dimensão baixa (<5 hp), discreto | ❌ 20 hp contínuos → explosão combinatória |
| **GA (Ultralytics)** | dimensão média, 30-100 trials, single-machine | ✅ ideal aqui |
| **Bayesian (Optuna, BOHB)** | sample-efficient, pode pausar | overkill p/ 30 trials, requer infra extra |
| **Ray Tune + ASHA** | paralelismo em cluster, centenas de trials | overkill p/ 2 GPUs |

### Configuração que usamos

* **30 iterações × 30 épocas/trial** (≈ 8h em 2× V100S no `small`)
* **Fitness**: combinação Ultralytics-padrão de `mAP50-95(B) + mAP50-95(M)`
* **20 hiperparâmetros** com ranges ajustados ao redor dos valores que
  funcionaram nas v2-v7 (não-defaults onde tínhamos sinal).
* **Hp fixos**: `optimizer="MuSGD"`, `amp=True`, `cos_lr=True`,
  `close_mosaic=15`, `erasing=0.0`, `batch=32`, `imgsz=640`. Esses já
  haviam sido ajustados nas ablations e não eram mais o gargalo.

> Por que trials curtos (30 ep) e não 120? Cada trial de 120 ep custaria
> ~1.5h × 30 = 45h. Em 30 ep o sinal já é suficiente para ranking — a
> validação em treino completo (§ 8) confirma se o ranking permanece.

## 4. Espaço de busca — 20 hiperparâmetros tunados

Os hp foram selecionados a partir das três famílias que mais variaram nas
ablations manuais v3-v7. Ranges centrados ao redor dos valores que vimos
funcionar, com folga para o GA explorar.

In [ ]:
SEARCH_SPACE = {
    # Aprendizado
    "lr0":             (5e-4, 5e-3),
    "lrf":             (0.005, 0.05),
    "momentum":        (0.85, 0.95, 0.3),    # 3-tupla = (lo, hi, gain)
    "weight_decay":    (0.0, 0.001),
    "warmup_epochs":   (0.0, 5.0),
    "warmup_momentum": (0.5, 0.95, 0.3),
    # Pesos das losses
    "box":             (3.0, 12.0),
    "cls":             (0.2, 1.5),
    "dfl":             (0.8, 3.0),
    # Augmentação fotométrica
    "hsv_h":           (0.0, 0.05),
    "hsv_s":           (0.0, 0.9),
    "hsv_v":           (0.0, 0.9),
    # Augmentação geométrica
    "degrees":         (0.0, 30.0),
    "translate":       (0.0, 0.3),
    "scale":           (0.2, 0.7),
    "fliplr":          (0.0, 0.6),
    "flipud":          (0.0, 0.3),
    # Augmentação de mixagem
    "mosaic":          (0.5, 1.0),
    "mixup":           (0.0, 0.3),
    "copy_paste":      (0.0, 0.3),
}

# Tabela bonita
rows = []
families = {
    "Aprendizado": ["lr0","lrf","momentum","weight_decay","warmup_epochs","warmup_momentum"],
    "Loss weights": ["box","cls","dfl"],
    "Augment fotométrica": ["hsv_h","hsv_s","hsv_v"],
    "Augment geométrica": ["degrees","translate","scale","fliplr","flipud"],
    "Augment de mixagem": ["mosaic","mixup","copy_paste"],
}
for fam, keys in families.items():
    for k in keys:
        rng = SEARCH_SPACE[k]
        rows.append({"Família": fam, "Hp": k, "Min": rng[0], "Max": rng[1]})
df_space = pd.DataFrame(rows)
df_space.style.set_properties(**{"text-align": "left"})


## 5. Convergência da busca

Carregamos `tune_results.csv` (uma linha por trial) e plotamos:

* a fitness real de cada trial (oscila com o GA explorando), e
* a envoltória `best so far` — a melhor fitness vista até a iteração.

A pergunta é: **o GA realmente melhorou ao longo das iterações, ou ficou
batendo no plateau cedo?**

In [ ]:
def load_tune_results() -> pd.DataFrame | None:
    if TUNE_NDJ.exists():
        rows = []
        with TUNE_NDJ.open() as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                r = json.loads(line)
                row = {"iteration": r["iteration"], "fitness": r["fitness"]}
                row.update(r.get("hyperparameters", {}))
                rows.append(row)
        return pd.DataFrame(rows).sort_values("iteration").reset_index(drop=True)
    if TUNE_CSV.exists():
        df = pd.read_csv(TUNE_CSV)
        df.insert(0, "iteration", range(1, len(df)+1))
        return df
    return None


df_tune = load_tune_results()
if df_tune is None:
    print("⚠ Nenhum tune_results encontrado — pule para §8 (validação) ou rode o tune primeiro.")
else:
    df_tune["best_so_far"] = df_tune["fitness"].cummax()
    print(f"Trials carregados: {len(df_tune)}")
    print(f"Best fitness     : {df_tune['fitness'].max():.4f} @ iter {int(df_tune['fitness'].idxmax())+1}")
    print(f"Std last 10      : {df_tune['fitness'].iloc[-10:].std():.4f}")
    df_tune.head()


In [ ]:
# Curva de convergência
if df_tune is not None:
    fig, ax = plt.subplots(figsize=(10, 4.6))
    ax.plot(df_tune["iteration"], df_tune["fitness"], "o-",
            color=OK_BLUE, alpha=0.55, lw=1.2, ms=6,
            label="fitness do trial")
    ax.plot(df_tune["iteration"], df_tune["best_so_far"], "-",
            color=OK_VERMILLION, lw=2.6, label="best so far")

    bi = int(df_tune["fitness"].idxmax())
    bx = df_tune.loc[bi, "iteration"]; by = df_tune["fitness"].max()
    ax.scatter([bx], [by], color=OK_VERMILLION, s=140, zorder=5,
               edgecolor="white", linewidth=1.8)
    ax.annotate(f"  best @ iter {int(bx)}\n  fitness = {by:.4f}",
                xy=(bx, by), xytext=(20, -42), textcoords="offset points",
                fontsize=10,
                bbox=dict(boxstyle="round,pad=0.5",
                          facecolor="#FFF8E5", edgecolor=OK_VERMILLION))

    ax.set_xlabel("Iteração do GA")
    ax.set_ylabel("Fitness  (Ultralytics — mAP50-95(B) + mAP50-95(M))")
    ax.set_title(f"Convergência do HPO — {len(df_tune)} trials × 30 ep cada")
    ax.grid(alpha=0.3); ax.legend(loc="lower right")
    plt.tight_layout(); plt.show()


**Interpretação típica esperada para este experimento**:

* O GA encontra o ótimo em torno do trial 9 (early exploitation) e os 21
  trials seguintes oscilam ao redor desse plateau.
* `improved last 25%? = False` — o sinal saturou. Isso pode significar:
  1. O search space está bem centrado nos valores corretos (a busca
     confirma rapidamente o ótimo local), ou
  2. O budget de 30 ep/trial não consegue distinguir configs medianas das
     boas (variância de validação domina diferenças sutis).

Em qualquer um dos casos, o resultado do trial 9 é confiável o bastante para
ser validado em treino full-length (§ 8).

## 6. O que aprendemos — correlação Pearson hp × fitness

Cada trial é um par (hp, fitness). Calculamos a correlação Pearson entre
cada hp e a fitness para responder: **quais hp realmente movem o ponteiro,
e quais o GA variou sem efeito mensurável?**

* `r > 0`: aumentar o hp tende a melhorar a fitness.
* `r < 0`: aumentar o hp tende a piorar a fitness.
* `|r| ≈ 0`: o hp não correlaciona com fitness — provavelmente irrelevante.

In [ ]:
if df_tune is not None:
    hp_cols = [c for c in df_tune.columns
               if c not in ("iteration","fitness","best_so_far")]
    corrs = df_tune[hp_cols + ["fitness"]].corr()["fitness"].drop("fitness")
    corrs = corrs.sort_values(key=abs, ascending=False)

    fig, ax = plt.subplots(figsize=(9, 6.5))
    colors = [OK_VERMILLION if v < -0.2 else OK_BLUE if v > 0.2 else OK_GREY
              for v in corrs.values]
    bars = ax.barh(range(len(corrs)), corrs.values, color=colors, edgecolor="white")
    ax.set_yticks(range(len(corrs)))
    ax.set_yticklabels(corrs.index, fontsize=10)
    ax.invert_yaxis()
    ax.axvline(0, color="black", lw=0.8)
    ax.axvline(+0.2, color=OK_GREY, ls=":", lw=0.8)
    ax.axvline(-0.2, color=OK_GREY, ls=":", lw=0.8)
    for i, v in enumerate(corrs.values):
        ax.text(v + (0.012 if v >= 0 else -0.012), i, f"{v:+.2f}",
                va="center", ha="left" if v >= 0 else "right",
                fontsize=9, fontweight="bold")
    ax.set_xlabel("Pearson r  (correlação hp × fitness)")
    ax.set_title("Hiperparâmetros que mais afetam a fitness")
    legend_handles = [
        mpatches.Patch(color=OK_VERMILLION, label="r < -0.2  (aumentar piora)"),
        mpatches.Patch(color=OK_BLUE,       label="r > +0.2  (aumentar melhora)"),
        mpatches.Patch(color=OK_GREY,       label="|r| < 0.2 (sem efeito)"),
    ]
    ax.legend(handles=legend_handles, loc="lower right", bbox_to_anchor=(1.0, -0.18), ncol=3)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout(); plt.show()

    print("\\nTOP 5 hp com efeito mais forte (|r| maior):")
    print(corrs.head(5).round(3).to_string())
    print("\\nHp que NÃO importam (|r| < 0.05) — descartáveis em busca futura:")
    print(corrs[corrs.abs() < 0.05].round(3).to_string())


**Achados típicos para este experimento**:

* `mixup` (r ≈ -0.77) e `copy_paste` (r ≈ -0.67) — efeitos negativos fortes.
  Isto **explica empiricamente** por que v3 e v4, que adicionaram essas
  augmentations agressivamente, performaram pior que v2.
* `lr0` (r ≈ -0.45) — confirma que learning rates baixos (~2e-3) são
  preferíveis aos altos (~5e-3) para fine-tuning.
* `translate`, `hsv_h`, `flipud` — efeitos positivos modestos (r ≈ +0.20).
* `scale`, `degrees`, `box`, `fliplr`, `warmup_momentum` — irrelevantes
  (|r| < 0.05). Em uma rodada futura, podem ser fixados nos defaults para
  reduzir a dimensão do espaço de busca.

## 7. Hiperparâmetros campeões vs default v7

Comparamos os hp que o GA encontrou (`best_hyperparameters.yaml`) contra o
default usado na v7 — para ver **quais variaram bastante**, o que dá
intuição sobre por que a config nova é melhor.

In [ ]:
DEFAULTS_V7 = {
    "lr0": 0.002,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "fliplr": 0.5,
    "flipud": 0.0,
    "mosaic": 1.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
}

best_hp = {}
if BEST_YAML.exists():
    with BEST_YAML.open() as f:
        best_hp = yaml.safe_load(f) or {}

if not best_hp:
    print("⚠ best_hyperparameters.yaml não encontrado — pule para §8.")
else:
    rows = []
    for k, dflt in DEFAULTS_V7.items():
        tuned = best_hp.get(k, np.nan)
        if dflt == 0:
            pct = ""
        else:
            pct = f"{(tuned - dflt) / dflt * 100:+.1f}%" if pd.notna(tuned) else ""
        rows.append({
            "Hp": k,
            "default v7": dflt,
            "tuned v8": tuned,
            "Δ absoluto": (tuned - dflt) if pd.notna(tuned) else np.nan,
            "Δ %": pct,
        })
    df_delta = pd.DataFrame(rows).set_index("Hp")
    print("Δ Hp campeão (v8) vs default v7:")
    df_delta.style.format({"default v7":"{:.5g}","tuned v8":"{:.5g}","Δ absoluto":"{:+.5g}"})


In [ ]:
# Heatmap visual do delta
if best_hp:
    # Δ relativo ao range do search space (cabe no [-1, +1])
    rel = []
    labels = []
    for k, dflt in DEFAULTS_V7.items():
        if k not in best_hp or k not in SEARCH_SPACE:
            continue
        rng = SEARCH_SPACE[k]
        span = max(rng[1] - rng[0], 1e-9)
        rel.append((best_hp[k] - dflt) / span)
        labels.append(k)
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = [OK_VERMILLION if v < 0 else OK_BLUE for v in rel]
    bars = ax.barh(range(len(rel)), rel, color=colors, edgecolor="white")
    ax.set_yticks(range(len(rel))); ax.set_yticklabels(labels, fontsize=10)
    ax.invert_yaxis(); ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Δ tuned − default, normalizado pelo range do search space")
    ax.set_title("O quanto cada hp se moveu em relação ao default v7\\n(barras grandes = a busca escolheu valor distinto do default)")
    for i, v in enumerate(rel):
        ax.text(v + (0.015 if v >= 0 else -0.015), i, f"{v:+.2f}",
                va="center", ha="left" if v >= 0 else "right",
                fontsize=9, fontweight="bold")
    ax.grid(axis="x", alpha=0.3); ax.set_xlim(-1.05, 1.05)
    plt.tight_layout(); plt.show()


**Observações típicas**:

* `weight_decay` cai 50× (5e-4 → 1e-5). Regularização L2 muito mais leve.
* `dfl` cai 26% (1.5 → 1.12). Distribution Focal Loss tem peso menor.
* `box` sobe 9% (7.5 → 8.19). Mais ênfase na regressão de bbox.
* `hsv_s` cai (0.70 → 0.56). Augmentação de saturação menos agressiva.
* `mixup` e `copy_paste` ficam em ~0 — confirma a evidência das correlações.

A interpretação geral: **o HPO encontrou uma config com regularização mais
leve, augmentação fotométrica mais sutil, e ZERO mistura entre amostras**
— oposto da direção que tínhamos tomado em v3/v4.

## 8. Validação em treino full-length

Trials de 30 ep podem ranquear errado se um hp parece bom em curto prazo
mas degenera em treino longo (ex: lr0 alto). Para confirmar, rodamos
**treino completo** (`epochs=120`, `patience=20`) com os hp campeões via
`train_with_tuned_hp.py --model small` (futuramente: `train_isic_2018_task_1_v8.py`).

In [ ]:
def load_results_csv(d: Path) -> pd.DataFrame | None:
    p = d / "results.csv"
    if not p.exists():
        return None
    df = pd.read_csv(p)
    df.columns = [c.strip() for c in df.columns]
    return df


df_full = load_results_csv(TUNED_DIR)
if df_full is None:
    print(f"⚠ Treino full-length não encontrado em {TUNED_DIR}. Rode primeiro o train_with_tuned_hp.py ou train_isic_2018_task_1_v8.py.")
else:
    M = "metrics/mAP50-95(M)"
    M50 = "metrics/mAP50(M)"
    bi = int(df_full[M].idxmax())
    print(f"Épocas treinadas: {len(df_full)} de 120")
    print(f"best mAP50-95(M) = {df_full[M].max():.4f} @ epoch {bi+1}")
    print(f"best mAP50(M)    = {df_full[M50].max():.4f}")
    print(f"último mAP50-95(M) = {df_full[M].iloc[-1]:.4f}  (early-stop por patience)")


In [ ]:
if df_full is not None:
    M = "metrics/mAP50-95(M)"; M50 = "metrics/mAP50(M)"
    fig, ax = plt.subplots(figsize=(10, 4.6))
    ax.plot(df_full["epoch"], df_full[M], "o-", color=OK_PURPLE,
            lw=1.6, ms=4.5, label="mAP50-95(M) — v8 tuned")
    ax.plot(df_full["epoch"], df_full[M].cummax(), "--", color=OK_VERMILLION,
            lw=2.0, label="best so far")

    bi = int(df_full[M].idxmax())
    ax.scatter([df_full.loc[bi,"epoch"]], [df_full[M].max()],
               color=OK_VERMILLION, s=140, zorder=5,
               edgecolor="white", linewidth=1.8)
    ax.annotate(f"  best @ ep {int(df_full.loc[bi,'epoch'])}\n  mAP50-95(M)={df_full[M].max():.4f}",
                xy=(df_full.loc[bi,"epoch"], df_full[M].max()),
                xytext=(20, -42), textcoords="offset points", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.5",
                          facecolor="#FFF8E5", edgecolor=OK_VERMILLION))

    # Baselines de referência
    BASELINES = {"v7 small (0.6964)": 0.6964, "v2 re-run (0.7169)": 0.7169}
    for lbl, y in BASELINES.items():
        ax.axhline(y, color=OK_GREY, ls=":", lw=1.3)
        ax.text(df_full["epoch"].max()+0.5, y, " "+lbl, va="center",
                fontsize=9, color="#444")

    ax.set_xlabel("Época"); ax.set_ylabel("mask mAP@50-95")
    ax.set_title("Treino full-length com hp campeões (v8 small)")
    ax.set_ylim(0.40, 0.78); ax.grid(alpha=0.3); ax.legend(loc="lower right")
    plt.tight_layout(); plt.show()


**Padrão observado no `small` (validação)**:

* Best mAP50-95(M) atingido em ~ep 9, depois oscila e decai levemente.
* `patience=20` triggera early-stop por volta de ep 29.
* Ganho de ~+1.5 pts mAP50-95(M) sobre v2 *re-run* (0.7169 → 0.7315).
* Bate todas as v3-v7 (~0.70) em **¼ do tempo de treino**.

Isso valida: **o ganho do HPO em 30-ep sobrevive ao treino completo.** A
config tuned (v8) é estável e reproduzível.

## 9. Comparação final — v2-v7 vs v8 (HPO-tuned)

Posicionamento da nova baseline contra todas as runs anteriores.

In [ ]:
# Construir tabela de comparação. Os valores históricos são fixos (literais),
# o do v8 é lido do TUNED_DIR (ou usa fallback se não existir).
HIST = [
    ("v4 small",        0.6484, 120, 24, OK_GREY),
    ("v7 small",        0.6964, 120, 46, OK_VERMILLION),
    ("v6 small",        0.6999, 120, 46, OK_GREY),
    ("v5 small",        0.7021, 120, 32, OK_GREY),
    ("v3 small",        0.7099, 120, 87, OK_GREY),
    ("v2-rerun small",  0.7169, 120, 47, OK_ORANGE),
    ("v2-orig small",   0.7368, 120, 47, OK_GREY),
]
v8_val = None
v8_eps = None
if df_full is not None:
    M = "metrics/mAP50-95(M)"
    v8_val = float(df_full[M].max())
    v8_eps = int(df_full.loc[df_full[M].idxmax(), "epoch"])
else:
    v8_val = 0.7315  # fallback do run que já fizemos
    v8_eps = 9

HIST.append(("v8 tuned small", v8_val, v8_eps, len(df_full) if df_full is not None else 29, OK_PURPLE))

df_comp = pd.DataFrame(HIST, columns=["run","mAP50-95(M)","best_epoch","epochs_run","color"])
df_comp = df_comp.sort_values("mAP50-95(M)").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5.2))
bars = ax.barh(df_comp["run"], df_comp["mAP50-95(M)"], color=df_comp["color"], edgecolor="white")
for i, (v, ep) in enumerate(zip(df_comp["mAP50-95(M)"], df_comp["best_epoch"])):
    ax.text(v + 0.003, i, f"  {v:.4f}  (best @ ep {ep})",
            va="center", fontsize=9, fontweight="bold")
ax.axvline(0.7169, color=OK_ORANGE, ls=":", lw=1.2)
ax.text(0.7169, len(df_comp)-0.3, " v2-rerun (baseline reproduzível)",
        rotation=90, va="top", ha="left", color=OK_ORANGE, fontsize=8)

ax.set_xlim(0.62, 0.76)
ax.set_xlabel("best mask mAP@50-95")
ax.set_title("v8 (HPO-tuned) vs runs manuais anteriores — small")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()


## 10. Conclusões e próximos passos

### O que aprendemos

1. **HPO encontrou ganho reproduzível** sobre a baseline (+1.5 pts mAP50-95(M)
   em treino full-length).
2. **mixup e copy_paste atrapalham** este dataset (correlação Pearson ≈ -0.7
   com fitness). Isso é evidência **estatística** — não anedótica — para
   reportar ao orientador.
3. **lr0 menor é melhor** para fine-tuning aqui (r = -0.45). Convergiu para
   ~2.3e-3.
4. **Vários hp não importam** (`scale`, `degrees`, `box`, `fliplr`,
   `warmup_momentum` — |r| < 0.05). Em rodadas futuras de busca, podem ser
   fixados.
5. **Modelo converge rápido com a nova config** — o best ocorre em ep ~9 de
   120. `patience=20` está bem dimensionado.

### Próximos passos

| Prioridade | Ação | Custo estimado | Ganho esperado |
|---|---|---|---|
| **Alta** | Treinar v8 nos 5 tamanhos (`train_isic_2018_task_1_v8.py`) | 5-8h | +1-2 pts em medium/large vs v7 |
| Média | Treinar HED com hp v8 | 2h × 5 = 10h | a verificar (pode haver ganho ortogonal) |
| Média | Tunar individualmente medium/large | 5-7h × 3 = ~20h | +0.5-1 pt em cada (especulativo) |
| Baixa | Refinar search space (remover hp |r|<0.05, ampliar mixup pra >0?) | 8h | marginal |
| Baixa | Trocar GA por Bayesian (Optuna) | infra extra | marginal pra <50 trials |

### Limitações do estudo

* **HPO feito apenas no small**. Aplicar os mesmos hp aos demais tamanhos é
  uma aproximação razoável mas não-validada.
* **30 trials × 30 ep** é budget modesto. Se quiséssemos rigor estatístico
  para publicação, faríamos N=50-100 trials e múltiplas seeds por config.
* **GA estagna cedo** — em ~iter 9. Random search ou Bayesian poderia
  encontrar pontos diferentes; vale comparar se o budget aumentar.
* **Validação com seed única**. Repetir o treino full-length com 3 seeds
  diferentes daria intervalo de confiança no ganho de +1.5 pts.

### O que reportar ao orientador

1. Não foi NAS (busca arquitetural). Foi HPO (busca de hiperparâmetros) — a
   distinção é importante para a literatura.
2. Resultado: **mAP50-95(M) = 0.7315** vs baseline 0.7169 (+1.5 pts), com
   um conjunto de hp **reproduzível e auditável** (`best_hyperparameters.yaml`).
3. Achado científico: mixup e copy_paste **ATRAPALHAM** neste dataset
   (evidência: Pearson r ≈ -0.7 sobre 30 trials).
4. Ferramentas: tudo no repo, scriptado, idempotente. Reproduzir leva ~8h
   em 2× V100S.